# Astrospheres and Solar-like Stellar Winds — Implementation
# 항성권과 태양형 항성풍 — 구현

**Paper**: Wood, B.E. (2004), *Living Rev. Solar Phys.*, 1, 2.

This notebook implements and visualizes key concepts from Wood's review:
이 노트북은 Wood 리뷰의 핵심 개념을 구현하고 시각화합니다:

1. **Heliosphere structure** — Termination shock, heliopause, bow shock radii / 태양권 구조
2. **Ly$\alpha$ absorption profile** — How ISM, heliospheric, and astrospheric components combine / Ly$\alpha$ 흡수 프로파일
3. **Charge exchange physics** — Hot hydrogen production / 전하 교환 물리
4. **Mass loss–activity relation** — $\dot{M} \propto F_X^{1.34}$ from Table 1 data / 질량 손실-활동성 관계
5. **Mass loss evolution law** — $\dot{M} \propto t^{-2.33}$ derivation chain / 질량 손실 진화 법칙
6. **Astrosphere size scaling** — How astrosphere size depends on wind strength / 항성권 크기 스케일링
7. **Solar wind history & Mars** — Implications for planetary atmosphere erosion / 태양풍 역사와 화성
8. **Connection to modern research** — Current state of the field / 현대 연구와의 연결

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from matplotlib.patches import FancyArrowPatch, Arc
from scipy.special import voigt_profile
from scipy.optimize import curve_fit

plt.rcParams.update({
    "figure.figsize": (10, 6),
    "font.size": 12,
    "axes.grid": True,
    "grid.alpha": 0.3,
})

## Part 1: Heliosphere Structure — Pressure Balance Model
## 파트 1: 태양권 구조 — 압력 균형 모델

The heliosphere's basic structure can be understood through **ram pressure balance** between the solar wind and the ISM. The termination shock (TS) radius is where solar wind ram pressure equals the total external pressure:

태양권의 기본 구조는 태양풍과 ISM 사이의 **ram pressure 균형**으로 이해할 수 있습니다. Termination shock (TS) 반경은 태양풍의 동압이 외부 총 압력과 같아지는 지점입니다:

$$P_{\rm sw}(r) = \frac{\dot{M} V_w}{4\pi r^2} = P_{\rm ISM} = \frac{1}{2}\rho_{\rm ISM} V_{\rm ISM}^2 + P_{\rm thermal} + P_{\rm mag}$$

Solving for the TS standoff distance:

$$R_{\rm TS} = \sqrt{\frac{\dot{M} V_w}{4\pi P_{\rm ISM}}}$$

The heliopause (HP) and bow shock (BS) distances can be estimated from the TS distance using typical ratios from hydrodynamic models: $R_{\rm HP}/R_{\rm TS} \approx 1.4$–$1.5$ and $R_{\rm BS}/R_{\rm TS} \approx 2.4$–$2.6$.

In [ ]:
# Physical constants
AU = 1.496e13       # cm
M_SUN = 1.989e33    # g
YR = 3.156e7        # s
KM = 1e5            # cm
M_P = 1.673e-24     # proton mass, g
K_B = 1.381e-16     # Boltzmann constant, erg/K

# Solar wind parameters
MDOT_SUN = 2e-14 * M_SUN / YR  # g/s, solar mass loss rate
V_SW = 400 * KM                 # cm/s, slow solar wind speed

# LISM parameters (from Wood 2004, Section 2.2)
N_HI_LISM = 0.14       # cm^-3, neutral hydrogen density
N_HP_LISM = 0.10        # cm^-3, proton density
T_LISM = 8000            # K
V_LISM = 25.7 * KM       # cm/s, LIC flow speed


def heliosphere_radii(mdot, v_wind, n_ism, v_ism, t_ism):
    """Calculate TS, HP, BS radii from pressure balance.
    
    Args:
        mdot: Mass loss rate (g/s).
        v_wind: Wind speed (cm/s).
        n_ism: ISM total number density (cm^-3).
        v_ism: ISM flow speed (cm/s).
        t_ism: ISM temperature (K).
    
    Returns:
        Dictionary with TS, HP, BS radii in AU.
    """
    # ISM ram pressure
    rho_ism = n_ism * M_P
    p_ram = 0.5 * rho_ism * v_ism**2
    
    # ISM thermal pressure (2nkT for fully ionized: ions + electrons)
    p_thermal = 2 * n_ism * K_B * t_ism
    
    # Total external pressure (ignoring magnetic pressure)
    p_total = p_ram + p_thermal
    
    # TS standoff distance from pressure balance
    r_ts = np.sqrt(mdot * v_wind / (4 * np.pi * p_total))
    
    # HP and BS from typical model ratios (Izmodenov et al. 2003)
    r_hp = 1.45 * r_ts
    r_bs = 2.5 * r_ts
    
    return {
        "TS": r_ts / AU,
        "HP": r_hp / AU,
        "BS": r_bs / AU,
    }


# Calculate for the Sun
n_total = N_HI_LISM + N_HP_LISM
radii = heliosphere_radii(MDOT_SUN, V_SW, n_total, V_LISM, T_LISM)
print("=== Solar Heliosphere Radii (Pressure Balance) ===")
print("=== 태양 태양권 반경 (압력 균형) ===")
for boundary, r in radii.items():
    print(f"  {boundary}: {r:.0f} AU")
print(f"\n  Voyager 1 TS crossing (2004): 94 AU")
print(f"  Model prediction matches well! / 모델 예측이 잘 맞습니다!")

In [ ]:
# Visualize heliosphere structure (schematic, inspired by Figure 3)
# 태양권 구조 시각화 (Figure 3 스타일)

fig, ax = plt.subplots(figsize=(12, 8))

# Draw boundaries as ellipses (asymmetric: compressed upwind, extended downwind)
theta = np.linspace(0, 2 * np.pi, 500)

def helio_boundary(r_nose, eccentricity, theta):
    """Parametric heliosphere boundary shape."""
    r = r_nose * (1 + eccentricity) / (1 + eccentricity * np.cos(theta))
    x = r * np.cos(theta)
    y = r * np.sin(theta)
    return x, y

# TS, HP, BS with increasing eccentricity (tail extends further)
for label, r_nose, ecc, color, ls in [
    ("Termination Shock (TS)", radii["TS"], 0.3, "blue", "--"),
    ("Heliopause (HP)", radii["HP"], 0.35, "green", "-"),
    ("Bow Shock (BS)", radii["BS"], 0.4, "red", "-."),
]:
    x, y = helio_boundary(r_nose, ecc, theta)
    ax.plot(x, y, color=color, linestyle=ls, linewidth=2, label=label)

# Sun at origin
ax.plot(0, 0, "o", color="gold", markersize=15, markeredgecolor="orange", zorder=5)
ax.annotate("Sun / 태양", (0, 0), (10, -20), fontsize=11,
            textcoords="offset points", ha="center")

# ISM flow direction
ax.annotate("", xy=(-radii["BS"] * 0.6, 0), xytext=(radii["BS"] * 1.3, 0),
            arrowprops=dict(arrowstyle="->", color="purple", lw=2))
ax.text(radii["BS"] * 1.0, 15, "LISM flow\n(25.7 km/s)", fontsize=10,
        color="purple", ha="center")

# Region labels
ax.text(0, radii["TS"] * 0.5, "Region 1\nSupersonic\nSolar Wind",
        ha="center", fontsize=9, color="blue", alpha=0.8)
ax.text(0, radii["HP"] * 0.85, "R2", ha="center", fontsize=9, color="green")
ax.text(radii["HP"] * 0.6, radii["BS"] * 0.8, "Region 3\nDisturbed ISM\n(Hydrogen Wall)",
        ha="center", fontsize=9, color="red", alpha=0.8)
ax.text(-radii["BS"] * 1.1, radii["BS"] * 0.8, "Region 4\nUndisturbed\nISM",
        ha="center", fontsize=9, color="gray")

ax.set_xlabel("Distance (AU) — Upwind → / 거리 (AU) — 바람 상류 →")
ax.set_ylabel("Distance (AU)")
ax.set_title("Schematic Heliosphere Structure / 태양권 구조 도식 (cf. Figure 3)")
ax.legend(loc="lower left", fontsize=10)
ax.set_aspect("equal")
ax.set_xlim(-400, 400)
ax.set_ylim(-350, 350)
plt.tight_layout()
plt.show()

## Part 2: Lyman-$\alpha$ Absorption Profile Simulation
## 파트 2: Lyman-$\alpha$ 흡수 프로파일 시뮬레이션

The key diagnostic in this paper is the stellar Ly$\alpha$ emission line as modified by absorption from multiple components along the line of sight (cf. Figure 6):

이 논문의 핵심 진단은 시선 방향의 여러 성분에 의해 변형되는 항성 Ly$\alpha$ 방출선입니다 (Figure 6 참조):

1. **Stellar chromospheric emission** — Gaussian-like profile / 항성 채층 방출
2. **ISM H I absorption** — broad, saturated, centered at ISM velocity / ISM H I 흡수
3. **ISM D I absorption** — narrow, offset by $-0.33$ Å ($-82$ km/s) / ISM D I 흡수
4. **Heliospheric absorption** — excess on the **red** side / 태양권 흡수 (적색 측)
5. **Astrospheric absorption** — excess on the **blue** side / 항성권 흡수 (청색 측)

We model each component using Voigt profiles and show how they combine.
각 성분을 Voigt profile로 모델링하고 결합하는 과정을 보여줍니다.

In [ ]:
# Ly-alpha absorption profile simulation (inspired by Figure 6)
# Ly-alpha 흡수 프로파일 시뮬레이션 (Figure 6 스타일)

C = 2.998e10         # speed of light, cm/s
LAMBDA_LYA = 1215.67  # Å, Lyman-alpha rest wavelength
M_H = M_P             # hydrogen mass


def velocity_to_wavelength(v_km_s):
    """Convert velocity (km/s) to wavelength offset from Lyα center."""
    return LAMBDA_LYA * (v_km_s / (C / KM))


def thermal_width(temperature, mass=M_H):
    """Doppler b-parameter (cm/s) for thermal broadening."""
    return np.sqrt(2 * K_B * temperature / mass)


def voigt_absorption(velocity, v_center, temperature, column_density,
                     mass=M_H, f_osc=0.4162):
    """Compute Voigt absorption optical depth profile.
    
    Args:
        velocity: Velocity array (km/s).
        v_center: Absorption center velocity (km/s).
        temperature: Gas temperature (K).
        column_density: Column density (cm^-2).
        mass: Particle mass (g).
        f_osc: Oscillator strength (0.4162 for Lyα).
    
    Returns:
        Optical depth as function of velocity.
    """
    # Doppler width
    b = thermal_width(temperature, mass) / KM  # km/s
    sigma = b / np.sqrt(2)  # Gaussian sigma in km/s
    
    # Natural damping parameter (Lyα: Γ = 6.27e8 s^-1)
    gamma = 6.27e8  # s^-1
    gamma_v = gamma * LAMBDA_LYA * 1e-8 / (C / KM)  # km/s equivalent
    
    # Voigt profile: H(a, u) where a = γ/(4πΔν_D), u = (v - v0)/b
    a_param = gamma_v / (2 * b)
    
    # Use scipy voigt_profile: voigt_profile(x, sigma, gamma)
    # where x = v - v_center, sigma is Gaussian sigma, gamma is Lorentzian HWHM
    profile = voigt_profile(velocity - v_center, sigma, gamma_v / 2)
    
    # Cross section: σ_0 = π e² f / (m_e c) * λ/c (in velocity space)
    # Simplified: optical depth = N * σ(v) where profile is normalized
    e_cgs = 4.803e-10  # esu
    m_e = 9.109e-28    # g
    sigma_0 = np.pi * e_cgs**2 * f_osc / (m_e * C)  # cm^2 * Hz
    # Convert to velocity: σ(v) = σ_0 * c/λ * profile(v) * (1e5 for km→cm)
    tau = column_density * sigma_0 * (C / (LAMBDA_LYA * 1e-8)) * profile * KM
    
    return tau


# Velocity grid (km/s) centered on Lyα
v = np.linspace(-200, 150, 2000)

# --- Stellar emission profile (Gaussian, representing chromosphere) ---
v_star = -18.0  # α Cen heliocentric velocity, km/s
emission_width = 80  # km/s
stellar_emission = 1.2 * np.exp(-0.5 * ((v - v_star) / emission_width)**2)

# --- ISM H I absorption ---
v_ism = -18.0   # km/s (ISM velocity toward α Cen)
T_ism = 5400    # K
logN_HI_ism = 17.8  # log column density (cm^-2), typical for α Cen
tau_ism = voigt_absorption(v, v_ism, T_ism, 10**logN_HI_ism)

# --- ISM D I absorption (offset by -82 km/s from H I) ---
v_DI = v_ism - 82  # D I offset
logN_DI = logN_HI_ism + np.log10(1.5e-5)  # D/H ~ 1.5e-5
tau_DI = voigt_absorption(v, v_DI, T_ism, 10**logN_DI, mass=2 * M_H)

# --- Heliospheric absorption (red side, ~30,000 K hot H) ---
v_helio = v_ism + 15  # redshifted relative to ISM
T_helio = 30000       # K (hydrogen wall)
logN_helio = 14.5     # much lower column
tau_helio = voigt_absorption(v, v_helio, T_helio, 10**logN_helio)

# --- Astrospheric absorption (blue side, ~30,000 K hot H) ---
v_astro = v_ism - 20  # blueshifted relative to ISM
T_astro = 30000       # K
logN_astro = 15.0     # α Cen astrospheric column
tau_astro = voigt_absorption(v, v_astro, T_astro, 10**logN_astro)

# Combine: observed = emission × exp(-τ_total)
tau_total_ism_only = tau_ism + tau_DI
tau_total_with_helio = tau_total_ism_only + tau_helio
tau_total_all = tau_total_with_helio + tau_astro

observed_ism_only = stellar_emission * np.exp(-tau_total_ism_only)
observed_with_helio = stellar_emission * np.exp(-tau_total_with_helio)
observed_all = stellar_emission * np.exp(-tau_total_all)

# --- Plot: Building up the absorption step by step ---
fig, axes = plt.subplots(2, 2, figsize=(14, 10))

# Panel 1: Stellar emission
ax = axes[0, 0]
ax.plot(v, stellar_emission, "k-", lw=2)
ax.set_title("1. Stellar Lyα Emission\n1. 항성 Lyα 방출")
ax.set_ylabel("Flux (arbitrary)")
ax.set_xlim(-180, 120)
ax.axvline(v_star, color="gray", ls=":", alpha=0.5, label=f"v_star = {v_star} km/s")
ax.legend(fontsize=9)

# Panel 2: After ISM absorption
ax = axes[0, 1]
ax.plot(v, stellar_emission, "k--", lw=1, alpha=0.4, label="Emission")
ax.plot(v, observed_ism_only, "g-", lw=2, label="After ISM (H I + D I)")
ax.set_title("2. + ISM Absorption\n2. + ISM 흡수")
ax.set_xlim(-180, 120)
ax.annotate("D I", xy=(v_DI, 0.05), fontsize=10, color="green", ha="center")
ax.annotate("H I (saturated)", xy=(v_ism, 0.02), fontsize=10, color="green", ha="center")
ax.legend(fontsize=9)

# Panel 3: Adding heliospheric absorption (red side)
ax = axes[1, 0]
ax.plot(v, observed_ism_only, "g--", lw=1, alpha=0.4, label="ISM only")
ax.plot(v, observed_with_helio, "r-", lw=2, label="+ Heliospheric (red side)")
ax.fill_between(v, observed_with_helio, observed_ism_only,
                where=observed_ism_only > observed_with_helio,
                alpha=0.3, color="red", label="Helio. excess")
ax.set_title("3. + Heliospheric Absorption (Red Side)\n3. + 태양권 흡수 (적색 측)")
ax.set_xlabel("Velocity (km/s)")
ax.set_ylabel("Flux (arbitrary)")
ax.set_xlim(-180, 120)
ax.legend(fontsize=9)

# Panel 4: Adding astrospheric absorption (blue side)
ax = axes[1, 1]
ax.plot(v, observed_with_helio, "r--", lw=1, alpha=0.4, label="ISM + Helio.")
ax.plot(v, observed_all, "b-", lw=2, label="+ Astrospheric (blue side)")
ax.fill_between(v, observed_all, observed_with_helio,
                where=observed_with_helio > observed_all,
                alpha=0.3, color="blue", label="Astro. excess")
ax.set_title("4. + Astrospheric Absorption (Blue Side)\n4. + 항성권 흡수 (청색 측)")
ax.set_xlabel("Velocity (km/s)")
ax.set_xlim(-180, 120)
ax.legend(fontsize=9)

fig.suptitle("Lyα Profile Build-up (cf. Figure 6) / Lyα 프로파일 구축 (Figure 6 비교)",
             fontsize=14, fontweight="bold", y=1.02)
plt.tight_layout()
plt.show()

## Part 3: Charge Exchange Physics — Hot Hydrogen Production
## 파트 3: 전하 교환 물리학 — 뜨거운 수소 생성

Charge exchange (CX) is the **key physical process** that makes astrospheric detection possible:

전하 교환(CX)은 항성권 검출을 가능하게 하는 **핵심 물리적 과정**입니다:

$$p^+_{\rm fast} + H_{\rm slow} \to H^*_{\rm fast} + p^+_{\rm slow}$$

The fast proton (heated by shock passage) captures an electron from a slow ISM neutral, producing a **fast (hot) neutral hydrogen atom** $H^*$. This hot H population:
- Has temperatures $T \sim 20{,}000$–$50{,}000$ K (from the shocked plasma)
- Accumulates in the "hydrogen wall" between HP and BS
- Produces detectable Ly$\alpha$ absorption

Below we simulate the CX process and the resulting hydrogen populations.

In [ ]:
# Charge exchange: simplified 1D model of hydrogen populations
# along the upwind direction through the heliosphere
# 전하 교환: 태양권 상류 방향을 따른 수소 집단의 단순화된 1D 모델

# Distance grid (AU), from Sun outward along upwind direction
r = np.linspace(1, 350, 1000)

# Boundary locations (from our pressure balance model)
r_ts = radii["TS"]
r_hp = radii["HP"]
r_bs = radii["BS"]

# --- Proton temperature profile (simplified) ---
def proton_temperature(r, r_ts, r_hp, r_bs):
    """Model proton temperature along upwind axis."""
    T = np.zeros_like(r)
    # Region 1: supersonic SW (adiabatic cooling ~ r^(-4/3) from ~1e6 K)
    mask1 = r < r_ts
    T[mask1] = 1e6 * (r[mask1] / 1)**(-4/3)
    T[mask1] = np.clip(T[mask1], 1e4, 1e6)
    # Region 2: shocked SW (heated at TS, ~1e6 K, then cooling)
    mask2 = (r >= r_ts) & (r < r_hp)
    frac = (r[mask2] - r_ts) / (r_hp - r_ts)
    T[mask2] = 1e6 * (1 - 0.3 * frac)
    # Region 3: shocked ISM (hot, ~1e5 K between HP and BS)
    mask3 = (r >= r_hp) & (r < r_bs)
    frac = (r[mask3] - r_hp) / (r_bs - r_hp)
    T[mask3] = 2e5 * (1 - 0.5 * frac)
    # Region 4: undisturbed ISM
    mask4 = r >= r_bs
    T[mask4] = T_LISM
    return T

# --- Neutral H density profile (simplified) ---
def neutral_h_density(r, r_ts, r_hp, r_bs):
    """Model neutral H density along upwind axis."""
    n = np.zeros_like(r)
    # Region 1: depleted by ionization (filtration)
    mask1 = r < r_ts
    n[mask1] = 0.04 * (r[mask1] / r_ts)**0.5  # heavily filtered
    # Region 2: some CX-produced hot H
    mask2 = (r >= r_ts) & (r < r_hp)
    frac = (r[mask2] - r_ts) / (r_hp - r_ts)
    n[mask2] = 0.04 + 0.06 * frac
    # Region 3: HYDROGEN WALL — peak density from CX
    mask3 = (r >= r_hp) & (r < r_bs)
    frac = (r[mask3] - r_hp) / (r_bs - r_hp)
    # Peak near HP, declining toward BS
    n[mask3] = 0.10 + 0.12 * np.exp(-3 * frac)  # peak ~ 0.22
    # Region 4: undisturbed ISM
    mask4 = r >= r_bs
    n[mask4] = N_HI_LISM
    return n

# --- Neutral H temperature profile ---
def neutral_h_temperature(r, r_ts, r_hp, r_bs):
    """Model neutral H temperature (heated by CX)."""
    T = np.ones_like(r) * T_LISM
    # Region 1: hot from CX with SW protons
    mask1 = r < r_ts
    T[mask1] = 5e4
    # Region 2: very hot from CX with shocked SW
    mask2 = (r >= r_ts) & (r < r_hp)
    T[mask2] = 1e5
    # Region 3: HYDROGEN WALL — heated by CX with shocked ISM
    mask3 = (r >= r_hp) & (r < r_bs)
    frac = (r[mask3] - r_hp) / (r_bs - r_hp)
    T[mask3] = 3e4 * np.exp(-2 * frac) + T_LISM
    # Region 4: cool ISM
    mask4 = r >= r_bs
    T[mask4] = T_LISM
    return T

T_proton = proton_temperature(r, r_ts, r_hp, r_bs)
n_H = neutral_h_density(r, r_ts, r_hp, r_bs)
T_H = neutral_h_temperature(r, r_ts, r_hp, r_bs)

# --- Plot ---
fig, axes = plt.subplots(3, 1, figsize=(12, 12), sharex=True)

# Boundary lines
for ax in axes:
    for pos, label, color in [(r_ts, "TS", "blue"), (r_hp, "HP", "green"), (r_bs, "BS", "red")]:
        ax.axvline(pos, color=color, ls="--", alpha=0.6)
        ax.text(pos, ax.get_ylim()[0] if ax.get_ylim()[0] > 0 else 0,
                label, color=color, fontsize=10, ha="center", va="bottom")

# Panel 1: Proton temperature
axes[0].semilogy(r, T_proton, "r-", lw=2)
axes[0].set_ylabel("Proton Temperature (K)\n양성자 온도")
axes[0].set_ylim(1e3, 2e6)
for pos, label, color in [(r_ts, "TS", "blue"), (r_hp, "HP", "green"), (r_bs, "BS", "red")]:
    axes[0].axvline(pos, color=color, ls="--", alpha=0.6, label=label)
axes[0].legend(loc="upper right")
axes[0].set_title("Heliosphere Profiles Along Upwind Direction / 상류 방향 태양권 프로파일")

# Panel 2: Neutral H density
axes[1].plot(r, n_H, "b-", lw=2)
axes[1].set_ylabel("Neutral H Density (cm⁻³)\n중성 H 밀도")
axes[1].axhline(N_HI_LISM, color="gray", ls=":", alpha=0.5, label=f"ISM: {N_HI_LISM} cm⁻³")
axes[1].fill_between(r, n_H, N_HI_LISM, where=(r >= r_hp) & (r < r_bs) & (n_H > N_HI_LISM),
                      alpha=0.3, color="orange", label="Hydrogen Wall")
for pos, label, color in [(r_ts, "TS", "blue"), (r_hp, "HP", "green"), (r_bs, "BS", "red")]:
    axes[1].axvline(pos, color=color, ls="--", alpha=0.6)
axes[1].legend()

# Panel 3: Neutral H temperature
axes[2].semilogy(r, T_H, "purple", lw=2)
axes[2].set_ylabel("Neutral H Temperature (K)\n중성 H 온도")
axes[2].set_xlabel("Distance from Sun (AU) / 태양으로부터 거리")
axes[2].axhline(T_LISM, color="gray", ls=":", alpha=0.5, label=f"ISM: {T_LISM} K")
axes[2].fill_between(r, T_H, T_LISM, where=(T_H > T_LISM * 1.5),
                      alpha=0.2, color="purple", label="CX-heated H")
for pos, label, color in [(r_ts, "TS", "blue"), (r_hp, "HP", "green"), (r_bs, "BS", "red")]:
    axes[2].axvline(pos, color=color, ls="--", alpha=0.6)
axes[2].legend()
axes[2].set_ylim(3e3, 2e5)

plt.tight_layout()
plt.show()

print("Key insight / 핵심 통찰:")
print("The hydrogen wall (between HP and BS) has BOTH elevated density AND")
print("temperature — this is what produces the detectable Lyα absorption.")
print("수소벽(HP와 BS 사이)은 밀도와 온도가 모두 높아져 검출 가능한 Lyα 흡수를 생성합니다.")

## Part 4: Mass Loss–Activity Relation — Reproducing Figure 14
## 파트 4: 질량 손실-활동성 관계 — Figure 14 재현

The central empirical result of this paper: mass loss rate (per unit surface area) correlates with coronal X-ray surface flux as a power law:

이 논문의 핵심 경험적 결과: 질량 손실률(단위 표면적당)이 코로나 X선 표면 플럭스와 멱법칙 관계를 보입니다:

$$\dot{M} \propto F_X^{1.34 \pm 0.18}$$

We reproduce Figure 14 using data from Table 1.

In [ ]:
# Data from Table 1 of Wood (2004, revised 2007)
# Columns: star name, spectral type, surface area (A_sun), log L_x, Mdot (Mdot_sun),
#          is_main_sequence, is_uncertain

stars = {
    # name: (surf_area_Asun, log_Lx, Mdot_Msun, is_MS, is_uncertain)
    "α Cen":     (2.22,  27.70, 2,     True,  False),
    "Prox Cen":  (0.023, 27.22, 0.2,   True,  False),  # upper limit
    "ε Eri":     (0.61,  28.32, 30,    True,  False),
    "61 Cyg A":  (0.46,  27.45, 0.5,   True,  False),
    "ε Ind":     (0.56,  27.39, 0.5,   True,  False),
    "EV Lac":    (0.123, 28.99, 1,     True,  False),
    "70 Oph":    (1.32,  28.49, 100,   True,  False),
    "36 Oph":    (0.88,  28.34, 15,    True,  False),
    "ξ Boo":     (1.00,  28.90, 5,     True,  False),
    "61 Vir":    (1.00,  26.87, 0.3,   True,  True),
    "δ Eri":     (6.66,  27.05, 4,     True,  False),
    "HD 128987": (0.71,  28.60, None,  True,  True),   # no Mdot
    "λ And":     (55,    30.53, 5,     False, True),
    "DK UMa":    (19.4,  30.36, 0.15,  False, True),
}

# Compute F_X (X-ray surface flux) and Mdot per unit surface area
L_SUN = 3.828e33  # erg/s
A_SUN = 4 * np.pi * (6.96e10)**2  # cm^2, solar surface area

star_names = []
log_fx_list = []
log_mdot_per_area_list = []
is_ms_list = []
is_uncertain_list = []

for name, (area, log_lx, mdot, is_ms, is_unc) in stars.items():
    if mdot is None:
        continue
    # X-ray surface flux: F_X = L_X / (4π R²) = L_X / (area * A_sun)
    lx = 10**log_lx
    fx = lx / (area * A_SUN)
    # Mass loss per unit surface area (in solar units)
    mdot_per_area = mdot / area
    
    star_names.append(name)
    log_fx_list.append(np.log10(fx))
    log_mdot_per_area_list.append(np.log10(mdot_per_area))
    is_ms_list.append(is_ms)
    is_uncertain_list.append(is_unc)

log_fx = np.array(log_fx_list)
log_mdot_pa = np.array(log_mdot_per_area_list)
is_ms = np.array(is_ms_list)
is_unc = np.array(is_uncertain_list)

# Fit power law to main sequence stars below saturation limit
# (exclude Prox Cen, EV Lac, ξ Boo which are above saturation, and evolved stars)
saturation_limit = np.log10(8e5)
fit_mask = is_ms & ~is_unc & (log_fx < saturation_limit + 0.5)
# Also exclude the three high-activity outliers
outlier_names = {"Prox Cen", "EV Lac", "ξ Boo"}
fit_mask2 = np.array([n not in outlier_names for n in star_names])
fit_mask = fit_mask & fit_mask2

def power_law(x, a, b):
    return a * x + b

popt, pcov = curve_fit(power_law, log_fx[fit_mask], log_mdot_pa[fit_mask])
slope, intercept = popt
slope_err = np.sqrt(pcov[0, 0])

print(f"Power law fit: Ṁ/A ∝ F_X^{slope:.2f} ± {slope_err:.2f}")
print(f"Paper value:   Ṁ/A ∝ F_X^1.34 ± 0.18")

# --- Plot: Reproducing Figure 14 ---
fig, ax = plt.subplots(figsize=(10, 8))

# Main sequence stars (filled circles)
ms_mask = is_ms & ~is_unc
ax.scatter(log_fx[ms_mask], log_mdot_pa[ms_mask], s=100, c="red",
           edgecolors="darkred", zorder=5, label="Main seq. (certain)")

# Uncertain/evolved (open circles)
unc_or_evolved = ~is_ms | is_unc
ax.scatter(log_fx[unc_or_evolved], log_mdot_pa[unc_or_evolved], s=100,
           facecolors="none", edgecolors="red", linewidths=2, zorder=5,
           label="Evolved / uncertain")

# Labels
for i, name in enumerate(star_names):
    offset = (8, 5) if name not in {"ε Ind", "61 Cyg A"} else (-5, -15)
    ax.annotate(name, (log_fx[i], log_mdot_pa[i]), fontsize=8,
                textcoords="offset points", xytext=offset)

# Power law fit line
fx_fit = np.linspace(3.5, 6.5, 100)
mdot_fit = slope * fx_fit + intercept
ax.plot(fx_fit, mdot_fit, "k-", lw=2, label=f"Fit: slope = {slope:.2f}")

# Uncertainty band
for s in [slope - slope_err, slope + slope_err]:
    mdot_band = s * fx_fit + intercept
    ax.plot(fx_fit, mdot_band, "k--", lw=0.5, alpha=0.5)
ax.fill_between(fx_fit,
                (slope - slope_err) * fx_fit + intercept,
                (slope + slope_err) * fx_fit + intercept,
                alpha=0.15, color="gray", label="Uncertainty band")

# Saturation line
ax.axvline(saturation_limit, color="orange", ls="-.", lw=2, alpha=0.7,
           label=f"Saturation: log F_X = {saturation_limit:.1f}")
ax.text(saturation_limit + 0.05, 2.0, "Saturation\nLine", fontsize=9,
        color="orange", rotation=90, va="center")

# Sun marker
# Sun: Mdot = 1 Msun, area = 1 Asun, so Mdot/A = 1
# log L_X ~ 27.3, F_X = 10^27.3 / A_sun
log_fx_sun = np.log10(10**27.3 / A_SUN)
ax.plot(log_fx_sun, 0, "o", color="gold", markersize=12,
        markeredgecolor="orange", markeredgewidth=2, zorder=6)
ax.annotate("Sun", (log_fx_sun, 0), fontsize=10, fontweight="bold",
            textcoords="offset points", xytext=(-20, -15))

ax.set_xlabel(r"$\log\, F_X$ (erg cm$^{-2}$ s$^{-1}$)")
ax.set_ylabel(r"$\log\, \dot{M}$ per unit surface area (solar units)")
ax.set_title("Mass Loss Rate vs X-ray Surface Flux (cf. Figure 14)\n"
             "질량 손실률 vs X선 표면 플럭스 (Figure 14 비교)")
ax.legend(loc="lower right", fontsize=9)
ax.set_xlim(3.5, 7.5)
ax.set_ylim(-2.5, 2.5)
plt.tight_layout()
plt.show()

## Part 5: Mass Loss Evolution Law — The Derivation Chain
## 파트 5: 질량 손실 진화 법칙 — 유도 사슬

The key result connecting stellar wind measurements to solar history is derived by chaining three empirical relations:

항성풍 측정을 태양 역사에 연결하는 핵심 결과는 세 가지 경험적 관계를 사슬로 연결하여 유도됩니다:

1. $\dot{M} \propto F_X^{1.34}$ — wind–activity (this paper)
2. $V_{\rm rot} \propto t^{-0.6}$ — rotation–age (Ayres 1997)
3. $F_X \propto V_{\rm rot}^{2.9}$ — activity–rotation

Combining: $\dot{M} \propto t^{-(0.6)(2.9)(1.34)} = t^{-2.33}$

In [ ]:
# Mass loss evolution law derivation and visualization
# 질량 손실 진화 법칙 유도 및 시각화

# Exponents from Wood (2004)
alpha_rot_age = -0.6    # V_rot ∝ t^α (Eq. 2)
beta_fx_rot = 2.9       # F_X ∝ V_rot^β (Eq. 3)
gamma_mdot_fx = 1.34    # Ṁ ∝ F_X^γ (Eq. 1)

# Combined exponent
combined_exponent = alpha_rot_age * beta_fx_rot * gamma_mdot_fx
print(f"=== Derivation Chain / 유도 사슬 ===")
print(f"V_rot ∝ t^{alpha_rot_age}")
print(f"F_X   ∝ V_rot^{beta_fx_rot}")
print(f"Ṁ     ∝ F_X^{gamma_mdot_fx}")
print(f"")
print(f"F_X ∝ t^({alpha_rot_age}×{beta_fx_rot}) = t^{alpha_rot_age * beta_fx_rot:.2f}")
print(f"Ṁ   ∝ t^({alpha_rot_age}×{beta_fx_rot}×{gamma_mdot_fx}) = t^{combined_exponent:.2f}")
print(f"")
print(f"Paper value: Ṁ ∝ t^(-2.33 ± 0.55)")

# Error propagation
dalpha = 0.1
dbeta = 0.3
dgamma = 0.18
# δ(αβγ) / (αβγ) = sqrt((δα/α)² + (δβ/β)² + (δγ/γ)²)
rel_err = np.sqrt((dalpha/abs(alpha_rot_age))**2 +
                   (dbeta/beta_fx_rot)**2 +
                   (dgamma/gamma_mdot_fx)**2)
err = abs(combined_exponent) * rel_err
print(f"Propagated uncertainty: ± {err:.2f}")

# --- Visualization: Three relations and their combination ---
fig, axes = plt.subplots(2, 2, figsize=(14, 10))
t_age = np.linspace(0.3, 10, 200)  # Gyr
t_sun = 4.6  # Gyr

# Panel 1: Rotation vs Age
ax = axes[0, 0]
v_rot = (t_age / t_sun)**alpha_rot_age  # normalized to Sun
ax.plot(t_age, v_rot, "b-", lw=2)
ax.axvline(t_sun, color="gold", ls="--", lw=2, alpha=0.7)
ax.axhline(1, color="gray", ls=":", alpha=0.5)
ax.set_xlabel("Age (Gyr)")
ax.set_ylabel(r"$V_{\rm rot}$ / $V_{\rm rot,\odot}$")
ax.set_title(f"Eq. 2: $V_{{rot}} \\propto t^{{{alpha_rot_age}}}$\n자전-나이 관계")
ax.annotate("Sun (4.6 Gyr)", (t_sun, 1), fontsize=10, color="orange",
            textcoords="offset points", xytext=(10, 10))

# Panel 2: X-ray flux vs Rotation
ax = axes[0, 1]
fx_norm = v_rot**beta_fx_rot  # normalized
ax.plot(v_rot, fx_norm, "r-", lw=2)
ax.set_xlabel(r"$V_{\rm rot}$ / $V_{\rm rot,\odot}$")
ax.set_ylabel(r"$F_X$ / $F_{X,\odot}$")
ax.set_title(f"Eq. 3: $F_X \\propto V_{{rot}}^{{{beta_fx_rot}}}$\nX선-자전 관계")
ax.set_xscale("log")
ax.set_yscale("log")
ax.plot(1, 1, "o", color="gold", markersize=10, markeredgecolor="orange", zorder=5)

# Panel 3: Mass loss vs X-ray flux
ax = axes[1, 0]
mdot_norm = fx_norm**gamma_mdot_fx
ax.plot(fx_norm, mdot_norm, "g-", lw=2)
ax.set_xlabel(r"$F_X$ / $F_{X,\odot}$")
ax.set_ylabel(r"$\dot{M}$ / $\dot{M}_\odot$")
ax.set_title(f"Eq. 1: $\\dot{{M}} \\propto F_X^{{{gamma_mdot_fx}}}$\n질량 손실-활동성 관계")
ax.set_xscale("log")
ax.set_yscale("log")
ax.plot(1, 1, "o", color="gold", markersize=10, markeredgecolor="orange", zorder=5)

# Panel 4: Combined — Mass loss vs Age (reproducing Figure 15)
ax = axes[1, 1]
mdot_vs_age = (t_age / t_sun)**combined_exponent  # Ṁ(t) / Ṁ_sun
# Uncertainty band
mdot_upper = (t_age / t_sun)**(combined_exponent + err)
mdot_lower = (t_age / t_sun)**(combined_exponent - err)

ax.fill_between(t_age, mdot_lower, mdot_upper, alpha=0.2, color="gray",
                label="Uncertainty band")
ax.plot(t_age, mdot_vs_age, "k-", lw=2,
        label=f"$\\dot{{M}} \\propto t^{{{combined_exponent:.2f}}}$")
ax.axvline(t_sun, color="gold", ls="--", lw=2, alpha=0.7)
ax.plot(t_sun, 1, "o", color="gold", markersize=12, markeredgecolor="orange",
        markeredgewidth=2, zorder=5, label="Sun (today)")

# Mark key epochs
for age, label in [(1.0, "1 Gyr"), (0.7, "0.7 Gyr\n(saturation)")]:
    mdot_at_age = (age / t_sun)**combined_exponent
    ax.plot(age, mdot_at_age, "rs", markersize=8)
    ax.annotate(f"{label}\n{mdot_at_age:.0f}× solar",
                (age, mdot_at_age), fontsize=9,
                textcoords="offset points", xytext=(15, -10))

# ξ Boo point (below the relation — saturation)
ax.plot(0.7, 5, "ko", markersize=10, label="ξ Boo (saturated)")

ax.set_xlabel("Age (Gyr) / 나이")
ax.set_ylabel(r"$\dot{M}$ / $\dot{M}_\odot$")
ax.set_title(f"Combined: $\\dot{{M}} \\propto t^{{{combined_exponent:.2f}}}$ (cf. Figure 15)\n"
             "질량 손실 진화 법칙 (Figure 15 비교)")
ax.set_yscale("log")
ax.legend(loc="upper right", fontsize=9)
ax.set_ylim(0.1, 500)

plt.tight_layout()
plt.show()

print(f"\nAt 1 Gyr: solar wind was ~{(1.0/t_sun)**combined_exponent:.0f}× stronger")
print(f"At 0.7 Gyr: solar wind was ~{(0.7/t_sun)**combined_exponent:.0f}× stronger")
print(f"1 Gyr에서: 태양풍은 현재보다 ~{(1.0/t_sun)**combined_exponent:.0f}배 강했음")
print(f"0.7 Gyr에서: 태양풍은 현재보다 ~{(0.7/t_sun)**combined_exponent:.0f}배 강했음")

## Part 6: Astrosphere Size Scaling
## 파트 6: 항성권 크기 스케일링

How does the astrosphere size depend on the wind strength? Using the pressure balance model, we can compute astrosphere radii for stars with different mass loss rates, reproducing the qualitative behavior shown in Figures 9 and 12.

항성풍의 세기에 따라 항성권 크기가 어떻게 달라지는지, 압력 균형 모델로 계산합니다. Figure 9과 12에 나타난 정성적 행동을 재현합니다.

$$R_{\rm TS} \propto \sqrt{\dot{M}}$$

Since the astrosphere is proportional to the TS radius, **doubling the mass loss rate increases the astrosphere radius by a factor of $\sqrt{2} \approx 1.4$**.

In [ ]:
# Astrosphere size comparison for stars from Table 1
# Table 1의 별들에 대한 항성권 크기 비교

# Stars with measured mass loss rates and ISM velocities from Table 1
astro_stars = {
    # name: (Mdot_solar, V_ISM km/s, angular_size_info)
    "ε Ind":    (0.5,  68),
    "61 Cyg A": (0.5,  86),
    "α Cen":    (2.0,  25),
    "ε Eri":    (30,   27),
    "36 Oph":   (15,   134),
    "70 Oph":   (100,  120),
    "λ And":    (5,    53),
}

fig, axes = plt.subplots(1, 2, figsize=(16, 7))

# --- Left panel: Astrosphere TS radii ---
ax = axes[0]
names = []
ts_radii = []
colors = []

for name, (mdot, v_ism) in astro_stars.items():
    r = heliosphere_radii(mdot * MDOT_SUN, V_SW, n_total, v_ism * KM, T_LISM)
    names.append(name)
    ts_radii.append(r["HP"])  # HP as proxy for astrosphere size
    colors.append("red" if name != "λ And" else "lightcoral")

# Sort by size
idx = np.argsort(ts_radii)[::-1]
names_sorted = [names[i] for i in idx]
radii_sorted = [ts_radii[i] for i in idx]
colors_sorted = [colors[i] for i in idx]

bars = ax.barh(range(len(names_sorted)), radii_sorted, color=colors_sorted,
               edgecolor="darkred", alpha=0.8)
ax.set_yticks(range(len(names_sorted)))
ax.set_yticklabels(names_sorted)
ax.set_xlabel("Heliopause Radius (AU)")
ax.set_title("Astrosphere Sizes from Pressure Balance\n항성권 크기 (압력 균형)")

# Add solar heliosphere for reference
ax.axvline(radii["HP"], color="gold", ls="--", lw=2, label=f"Sun HP: {radii['HP']:.0f} AU")
ax.legend()

for i, r in enumerate(radii_sorted):
    ax.text(r + 5, i, f"{r:.0f} AU", va="center", fontsize=9)

# --- Right panel: Astrosphere size vs mass loss rate ---
ax = axes[1]
mdot_range = np.logspace(-1, 2.5, 100)  # 0.1 to 300 Mdot_sun
hp_range = []
for m in mdot_range:
    r = heliosphere_radii(m * MDOT_SUN, V_SW, n_total, V_LISM, T_LISM)
    hp_range.append(r["HP"])

ax.loglog(mdot_range, hp_range, "b-", lw=2, label=r"$R_{\rm HP} \propto \sqrt{\dot{M}}$")

# Plot individual stars
for name, (mdot, v_ism) in astro_stars.items():
    r = heliosphere_radii(mdot * MDOT_SUN, V_SW, n_total, v_ism * KM, T_LISM)
    marker = "o" if name != "λ And" else "s"
    ax.plot(mdot, r["HP"], marker, color="red", markersize=8, zorder=5)
    ax.annotate(name, (mdot, r["HP"]), fontsize=8,
                textcoords="offset points", xytext=(5, 5))

# Sun
ax.plot(1, radii["HP"], "o", color="gold", markersize=12,
        markeredgecolor="orange", markeredgewidth=2, zorder=6)
ax.annotate("Sun", (1, radii["HP"]), fontsize=10, fontweight="bold",
            textcoords="offset points", xytext=(8, -12))

# Full Moon angular size comparison
# At d = 5 pc, 1000 AU subtends ~0.2 arcsec. Full Moon ~ 1800 arcsec.
# ε Eri at 3.2 pc with R_HP ~ 3000 AU → angular size ~ 3000/(3.2*206265) ~ 4.5 arcsec
# 70 Oph at 5.1 pc with R_HP ~ 1500 AU → angular size ~ 1500/(5.1*206265) ~ 1.4 arcsec
# Full Moon is 1800 arcsec → these are NOT comparable to full moon.
# But Wood says the LARGEST astrospheres (ε Eri at ~3000 AU, 70 Oph at ~7000 AU)
# could be comparable... let's check with his higher Mdot values

ax.set_xlabel(r"$\dot{M}$ / $\dot{M}_\odot$")
ax.set_ylabel("Heliopause Radius (AU)")
ax.set_title(r"Astrosphere Size vs Mass Loss Rate" + "\n" +
             r"항성권 크기 vs 질량 손실률")
ax.legend(fontsize=10)

plt.tight_layout()
plt.show()

print("Note: Stars with stronger winds (ε Eri, 70 Oph) have astrospheres")
print("that would subtend several arcseconds — comparable to or larger than")
print("the angular resolution of HST, though not visible in direct imaging.")
print("참고: 강한 바람의 별(ε Eri, 70 Oph)의 항성권은 HST 각분해능과 비슷하거나")
print("더 큰 각도를 차지하지만, 직접 영상으로는 보이지 않습니다.")

## Part 7: Solar Wind History and Mars Atmosphere Erosion
## 파트 7: 태양풍 역사와 화성 대기 침식

One of the most compelling implications of the mass loss evolution law: Mars lost its global magnetic field ~3.9 Gyr ago (Acuña et al. 1999), after which it was exposed to a solar wind that was **~80× stronger** than today's. This may have been a major driver of Martian atmospheric loss.

질량 손실 진화 법칙의 가장 설득력 있는 함의: 화성은 ~39억 년 전에 전구 자기장을 잃었고(Acuña et al. 1999), 그 이후 현재보다 **~80배 강한** 태양풍에 노출되었습니다. 이것이 화성 대기 손실의 주요 원인이었을 가능성이 있습니다.

In [ ]:
# Solar wind history timeline with planetary implications
# 태양풍 역사 타임라인과 행성 관련 함의

fig, ax = plt.subplots(figsize=(14, 8))

# Time axis: age of Sun from formation (0 = birth, 4.6 = now)
t_sun_age = 4.6  # Gyr
t_history = np.linspace(0.3, t_sun_age, 500)
# Solar wind strength relative to today
mdot_history = (t_history / t_sun_age)**combined_exponent

# Uncertainty band
mdot_upper = (t_history / t_sun_age)**(combined_exponent + err)
mdot_lower = (t_history / t_sun_age)**(combined_exponent - err)

ax.fill_between(t_history, mdot_lower, mdot_upper, alpha=0.15, color="gray")
ax.semilogy(t_history, mdot_history, "b-", lw=3,
            label=r"$\dot{M}(t) \propto t^{-2.33}$")

# Key events
events = [
    (0.7, "Saturation limit\n포화 한계", "red", "v"),
    (0.8, "Mars: earliest evidence\nof liquid water\n화성 최초 액체물 증거", "brown", "s"),
    (3.9, "Mars loses\nmagnetic field\n화성 자기장 상실", "darkred", "D"),
    (4.6, "Today / 현재", "gold", "o"),
]

for t_event, label, color, marker in events:
    mdot_at_event = (t_event / t_sun_age)**combined_exponent
    ax.plot(t_event, mdot_at_event, marker, color=color, markersize=12,
            markeredgecolor="black", markeredgewidth=1.5, zorder=5)
    ax.annotate(f"{label}\n({mdot_at_event:.0f}× today)",
                (t_event, mdot_at_event), fontsize=9, color=color,
                textcoords="offset points", xytext=(15, 10),
                arrowprops=dict(arrowstyle="->", color=color, lw=1.5),
                bbox=dict(boxstyle="round,pad=0.3", facecolor="white", alpha=0.8))

# Highlight the Mars exposure period
ax.axvspan(0.7, 4.6, alpha=0.05, color="red")
ax.axvline(0.7, color="red", ls=":", alpha=0.5)

# Annotations
ax.annotate("Mars exposed to strong solar wind\n(no magnetic shield)\n"
            "화성이 강한 태양풍에 노출\n(자기 방패 없음)",
            xy=(2.5, 10), fontsize=11, ha="center", color="darkred",
            bbox=dict(boxstyle="round", facecolor="lightyellow", alpha=0.8))

# Cumulative mass loss calculation
# ∫ Ṁ dt from 0.7 to 4.6 Gyr: Ṁ₀ ∫ (t/4.6)^-2.33 dt
# = Ṁ₀ × 4.6^2.33 × ∫ t^-2.33 dt from 0.7 to 4.6
# = Ṁ₀ × 4.6^2.33 × [t^-1.33 / (-1.33)] from 0.7 to 4.6
from scipy.integrate import quad
def mdot_func(t):
    return (t / t_sun_age)**combined_exponent
total_mass_lost, _ = quad(mdot_func, 0.7, 4.6)
total_mass_solar = total_mass_lost * MDOT_SUN * YR * 1e9 / M_SUN

ax.text(0.95, 0.95, f"Cumulative mass lost (0.7–4.6 Gyr):\n"
        f"누적 질량 손실:\n"
        f"~{total_mass_solar*100:.1f}% M☉ ({total_mass_solar:.4f} M☉)\n"
        f"→ Insufficient to solve Faint Young Sun\n"
        f"→ 어린 희미한 태양 문제 해결에 불충분",
        transform=ax.transAxes, fontsize=10, va="top", ha="right",
        bbox=dict(boxstyle="round", facecolor="lightyellow", alpha=0.9))

ax.set_xlabel("Solar Age (Gyr) / 태양 나이", fontsize=12)
ax.set_ylabel(r"$\dot{M}$ / $\dot{M}_{\odot,\rm today}$", fontsize=12)
ax.set_title("Solar Wind History & Planetary Implications\n"
             "태양풍 역사와 행성 함의 (cf. Figure 15, Section 5.3–5.4)")
ax.legend(loc="upper left", fontsize=11)
ax.set_xlim(0.3, 5.0)
ax.set_ylim(0.5, 500)
plt.tight_layout()
plt.show()

## Part 8: Summary — Connection to Modern Research
## 파트 8: 요약 — 현대 연구와의 연결

| This Paper (2004/2007) | Modern Status (2020s) | 현대 상태 |
|---|---|---|
| ~13 astrospheric detections | ~20+ detections with HST/STIS (repaired 2009) | HST/STIS 복구(2009) 후 20+ 검출 |
| $\dot{M} \propto F_X^{1.34}$ | Revised with more data; saturation regime better characterized | 더 많은 데이터로 수정; 포화 영역 더 잘 특성화 |
| Voyager 1 at TS (94 AU) | Voyager 1 & 2 in interstellar space; IBEX & IMAP mapping the heliosphere | Voyager 1 & 2 성간 공간; IBEX & IMAP 태양권 매핑 |
| 2D axisymmetric models | 3D MHD models with neutrals (Opher, Pogorelov groups) | 중성수소 포함 3D MHD 모델 |
| No direct imaging of astrospheres | Still no direct imaging; Lyα absorption remains primary technique | 여전히 직접 영상 없음; Lyα 흡수가 주요 기법 |
| Mars atmosphere erosion hypothesis | MAVEN mission (2014+) confirming ongoing atmospheric loss | MAVEN 미션(2014+)이 진행 중인 대기 손실 확인 |

This review remains a foundational reference for understanding stellar winds in the solar context.
이 리뷰는 태양 맥락에서 항성풍을 이해하는 데 기초적인 참고 문헌으로 남아 있습니다.